# Process and Plot Satellite Chlorophyll Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [ ]:
# Read the CSV file, convert date column to datetime, and sort by date
def read_data(inpath):
    df = pd.read_csv(inpath)
    # Ensure date parsing
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df.sort_values('date', inplace=True)
    return df

# Convert NDCI to chloropyll (Landsat and Sentinel only)
def NDCI_to_chlorophyll(df, slope, offset):
    df['chl'] = df['ndci']*slope + offset
    return df

# Remove outliers using the Median Absolute Deviation method
def remove_outliers(df, value_col='chl', threshold=5.0):
    median = df[value_col].median()
    mad = np.median(np.abs(df[value_col] - median))
    # Convert MAD to robust standard deviation
    if mad == 0:
        # If MAD is zero (flat series), no outliers
        return df.copy()
    robust_sd = 1.4826 * mad
    z_scores = np.abs(df[value_col] - median) / robust_sd
    cleaned_df = df[z_scores < threshold].copy()
    return cleaned_df

# Plot the time series
def plot_data(df, label, title, ylabel):
    fig, axes = plt.subplots(1, 1, figsize=(12, 6), sharex=True)
    marker = ''
    linestyle = '-'
    # linecolor = 'deepskyblue'
    rgb = (60, 50, 255)
    linecolor = '#{:02X}{:02X}{:02X}'.format(*rgb)
    axes.plot(df['date'], df['chl'], marker=marker, linestyle=linestyle, color=linecolor, label=label)
    axes.set_title(title)
    axes.set_ylabel(ylabel)
    axes.grid(True)
    fig.tight_layout()
    return axes

## Landsat

In [ ]:
# Read the CSV file
df = read_data('Detroit_NDCI_Landsat_2011-2025.csv')

# Compute chlorophyll
offset = 0
slope = 80
df = NDCI_to_chlorophyll(df, slope, offset)

# Clean dataset
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Plot data
label = 'Landsat Chl-a'
title = 'Landsat Chlorophyll Data'
ylabel = 'Chl-a (µg/L)'
axes = plot_data(df, label, title, ylabel)

## Sentinel


In [ ]:
# Read the CSV file
df = read_data('Detroit_NDCI_Sentinel2_2011-2025.csv')

# Compute chlorophyll
offset = 0
slope = 80
df = NDCI_to_chlorophyll(df, slope, offset)

# Clean dataset
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Plot data
label = 'Sentinel Chl-a'
title = 'Sentinel Chlorophyll Data'
ylabel = 'Chl-a (µg/L)'
plot_data(df, label, title, ylabel)

## MODIS

In [ ]:
# Read the CSV files
aqua_df = read_data('Detroit_MODIS_Aqua_500m_Chl_singlePixel.csv')
terra_df = read_data('Detroit_MODIS_Terra_500m_Chl_singlePixel.csv')

# Clean the datasets
aqua_df_clean = remove_outliers(aqua_df, 'chl', threshold=5.0)
terra_df_clean = remove_outliers(terra_df, 'chl', threshold=5.0)

# Plot MODIS-Aqua
label = 'MODIS-Aqua Chl-a'
title = 'MODIS-Aqua Chlorophyll Data'
ylabel = 'Chl-a (µg/L)'
plot_data(aqua_df_clean, label, title, ylabel)

# Plot MODIS-Terra
label = 'MODIS-Terra Chl-a'
title = 'MODIS-Terra Chlorophyll Data'
ylabel = 'Chl-a (µg/L)'
plot_data(terra_df_clean, label, title, ylabel)